# IATItt

## Download the CRS parquet file

Uses `bulk_download_crs()` from [oda_reader](https://github.com/ONEcampaign/oda_reader). It looks up the current `CRS-Parquet` file in the OECD's dataflow metadata, then downloads it (about 1.18 GB). Pass `reduced_version=True` for the smaller file (about 296 MB, with fewer columns).

In [ ]:
%pip install oda_reader matplotlib

In [ ]:
from pathlib import Path

from oda_reader import bulk_download_crs
from oda_reader.exceptions import BulkDownloadChallengeError

# Kept outside the Google Drive folder so Insync doesn't sync the file
DATA_DIR = Path.home() / "oda_data"
DATA_DIR.mkdir(exist_ok=True)
CRS_PATH = DATA_DIR / "CRS.parquet"

if CRS_PATH.exists():
    print(f"Already downloaded: {CRS_PATH}")
else:
    try:
        bulk_download_crs(save_to_path=DATA_DIR)
    except BulkDownloadChallengeError as e:
        raise RuntimeError(
            f"Download blocked by Cloudflare, so {CRS_PATH} was not created.\n"
            f"Open {e.url} in a browser, save the file to that path, "
            "then re-run this cell."
        ) from e

**If the download is blocked:** the OECD's file server can challenge scripted requests, but browser downloads are supported. Open the URL printed above in a browser and save the file to `~/oda_data/CRS.parquet`. The cell above then skips the download.

## Load the data

Column names in this file are snake_case (`donor_code`, `crs_id`). The full file needs several GB of RAM.

In [ ]:
import pandas as pd

crs = pd.read_parquet(CRS_PATH)
print(crs.shape)
crs.head()

In [ ]:
# Germany (donor 5), Water Supply & Sanitation (sector 140), 2015 onwards
filtered = crs[
    (crs["donor_code"] == 5) & (crs["year"] > 2014) & (crs["sector_code"] == 140)
]

# Disbursements by agency and year, in USD million (constant prices)
table = filtered.pivot_table(
    index="agency_name",
    columns="year",
    values="usd_disbursement_defl",
    aggfunc="sum",
    fill_value=0,
)
table["Total"] = table.sum(axis=1)
table = table.sort_values("Total", ascending=False)

# Group agencies with a total under USD 10 million as "Other"
small = table[table["Total"] < 10]
table = table[table["Total"] >= 10].copy()
table.loc["Other"] = small.sum()

table.loc["Total"] = table.sum()
table.round(0).astype(int)

In [ ]:
import matplotlib.pyplot as plt

# Same data as the table above, without its totals
chart = table.drop(index="Total", columns="Total")
years = chart.columns.astype(int)

colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
colors = colors[: len(chart)]

fig, ax = plt.subplots(figsize=(11, 8))

# Each band is one agency; each line is the running total up to that agency
ax.stackplot(years, chart.values, colors=colors, alpha=0.25, linewidth=0)
for top, color, label in zip(chart.cumsum().values, colors, chart.index):
    ax.plot(years, top, color=color, linewidth=2, label=label)

ax.set_title(
    "German disbursements for water supply and sanitation by agency, 2015–2024",
    loc="left", fontsize=14, fontweight="bold", color="#0b0b0b", pad=14,
)
ax.set_ylabel("USD million, constant prices", color="#52514e")
ax.set_xlim(years.min(), years.max())
ax.set_xticks(years)
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
ax.tick_params(colors="#52514e", length=0)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")

# Legend below the chart to fit the full agency names, in stack order (top first)
handles, legend_labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1], legend_labels[::-1],
    loc="upper left", bbox_to_anchor=(0, -0.07), frameon=False, labelcolor="#52514e",
)

fig.tight_layout()
plt.show()